In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import json
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

CATALOG = "toxictide"
GOLD = f"{CATALOG}.gold"

site_daily_sdf = spark.read.table(f"{GOLD}.site_daily_features")
site_daily_pdf = site_daily_sdf.toPandas()

print("Loaded site_daily_features")
print("rows =", len(site_daily_pdf))
print("cols =", len(site_daily_pdf.columns))
print(site_daily_pdf.columns.tolist())

Loaded site_daily_features
rows = 270
cols = 44
['site_id', 'calendar_date', 'ca_beach__source_obs_date', 'ca_beach__freshness_days', 'ca_beach__enterococcus', 'ca_beach__total_coliform', 'ca_beach__fecal_coliform', 'ca_beach__min_distance_km', 'ca_beach__obs_count', 'tides__source_obs_date', 'tides__freshness_days', 'tides__water_level', 'tides__sigma', 'tides__i', 'tides__l', 'tides__min_distance_km', 'tides__obs_count', 'site_name', 'site_type', 'region_name', 'lat_dec', 'lon_dec', 'site_priority', 'notes', 'lat_idx', 'lon_idx', 'tile_id', 'default_match_radius_km', 'calcofi__depthm', 'calcofi__t_degc', 'calcofi__salnty', 'calcofi__o2ml_l', 'calcofi__stheta', 'calcofi__o2sat', 'calcofi__chlora', 'calcofi__po4um', 'calcofi__sio3um', 'calcofi__no2um', 'calcofi__no3um', 'calcofi__obs_count', 'calcofi__avg_distance_km', 'ca_beach__available', 'tides__available', 'n_sources_available']


In [0]:
aquaculture_pdf = site_daily_pdf[site_daily_pdf["site_type"] == "aquaculture"].copy()
beach_pdf = site_daily_pdf[site_daily_pdf["site_type"] == "beach"].copy()

print("aquaculture rows =", len(aquaculture_pdf))
print("beach rows =", len(beach_pdf))
print("aquaculture sites =", aquaculture_pdf["site_id"].nunique())
print("beach sites =", beach_pdf["site_id"].nunique())

aquaculture rows = 180
beach rows = 90
aquaculture sites = 6
beach sites = 3


In [0]:
def numeric_cols(df: pd.DataFrame, exclude=None):
    exclude = set(exclude or [])
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols


def available_prefixed_cols(df: pd.DataFrame, prefix: str, exclude_suffixes=None):
    exclude_suffixes = exclude_suffixes or []
    cols = []
    for c in df.columns:
        if not c.startswith(prefix):
            continue
        if any(c.endswith(sfx) for sfx in exclude_suffixes):
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols


def quantile_scale(s: pd.Series, lower_q=0.1, upper_q=0.9, invert=False):
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)

    lo = x.quantile(lower_q)
    hi = x.quantile(upper_q)

    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        out = pd.Series(np.nan, index=s.index)
    else:
        out = ((x - lo) / (hi - lo)).clip(0, 1)

    if invert:
        out = 1 - out

    return out


def deviation_scale(s: pd.Series, lower_q=0.1, upper_q=0.9):
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)

    med = x.median()
    dev = (x - med).abs()

    lo = dev.quantile(lower_q)
    hi = dev.quantile(upper_q)

    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(np.nan, index=s.index)

    return ((dev - lo) / (hi - lo)).clip(0, 1)


def mean_available(df: pd.DataFrame, cols: list[str], out_col: str):
    if not cols:
        df[out_col] = np.nan
        return df
    df[out_col] = df[cols].mean(axis=1, skipna=True)
    return df


def weighted_mean_available(df: pd.DataFrame, cols: list[str], weights: list[float], out_col: str):
    if not cols:
        df[out_col] = np.nan
        return df

    numer = np.zeros(len(df), dtype=float)
    denom = np.zeros(len(df), dtype=float)

    for c, w in zip(cols, weights):
        vals = pd.to_numeric(df[c], errors="coerce")
        mask = vals.notna().astype(float)
        numer += vals.fillna(0).to_numpy() * w
        denom += mask.to_numpy() * w

    out = np.where(denom > 0, numer / denom, np.nan)
    df[out_col] = out
    return df


def risk_level_from_score(score: pd.Series):
    return pd.cut(
        score,
        bins=[-np.inf, 0.35, 0.55, 0.75, np.inf],
        labels=["low", "moderate", "high", "severe"],
    ).astype("string")


def top_driver_json(row, driver_map: dict):
    ranked = sorted(
        [(k, float(v)) for k, v in driver_map.items() if pd.notna(v)],
        key=lambda x: x[1],
        reverse=True,
    )
    return json.dumps(
        [{"driver": k, "score": round(v, 4)} for k, v in ranked[:3]]
    )


def to_spark_and_save(pdf: pd.DataFrame, table_name: str):
    out_pdf = pdf.where(pd.notnull(pdf), None)
    sdf = spark.createDataFrame(out_pdf)
    sdf.write.format("delta").mode("overwrite").saveAsTable(table_name)
    return sdf


try:
    mlflow.set_experiment("/Shared/toxictide_hackathon")
except Exception as e:
    print("Could not set shared experiment, using notebook default:", e)

2026/04/19 17:52:20 INFO mlflow.tracking.fluent: Experiment with name '/Shared/toxictide_hackathon' does not exist. Creating a new experiment.


In [0]:
prefix_summary = {
    "calcofi": [c for c in site_daily_pdf.columns if c.startswith("calcofi__")],
    "ca_beach": [c for c in site_daily_pdf.columns if c.startswith("ca_beach__")],
    "tides": [c for c in site_daily_pdf.columns if c.startswith("tides__")],
    "cce": [c for c in site_daily_pdf.columns if c.startswith("cce__")],
    "cdip": [c for c in site_daily_pdf.columns if c.startswith("cdip__")],
    "nws": [c for c in site_daily_pdf.columns if c.startswith("nws__")],
}

for k, v in prefix_summary.items():
    print(f"{k}: {len(v)} cols")
    print(v)
    print()

calcofi: 13 cols
['calcofi__depthm', 'calcofi__t_degc', 'calcofi__salnty', 'calcofi__o2ml_l', 'calcofi__stheta', 'calcofi__o2sat', 'calcofi__chlora', 'calcofi__po4um', 'calcofi__sio3um', 'calcofi__no2um', 'calcofi__no3um', 'calcofi__obs_count', 'calcofi__avg_distance_km']

ca_beach: 8 cols
['ca_beach__source_obs_date', 'ca_beach__freshness_days', 'ca_beach__enterococcus', 'ca_beach__total_coliform', 'ca_beach__fecal_coliform', 'ca_beach__min_distance_km', 'ca_beach__obs_count', 'ca_beach__available']

tides: 9 cols
['tides__source_obs_date', 'tides__freshness_days', 'tides__water_level', 'tides__sigma', 'tides__i', 'tides__l', 'tides__min_distance_km', 'tides__obs_count', 'tides__available']

cce: 0 cols
[]

cdip: 0 cols
[]

nws: 0 cols
[]



In [0]:
aq = aquaculture_pdf.copy()

# ---------- component 1: biogeochemistry / bloom predisposition ----------
bio_scaled_cols = []

for c in ["calcofi__chlora", "calcofi__no3um", "calcofi__po4um", "calcofi__sio3um"]:
    if c in aq.columns:
        sc = f"{c}__scaled"
        aq[sc] = quantile_scale(aq[c])
        bio_scaled_cols.append(sc)

oxygen_scaled_cols = []
for c in ["calcofi__o2ml_l", "calcofi__o2sat"]:
    if c in aq.columns:
        sc = f"{c}__invscaled"
        aq[sc] = quantile_scale(aq[c], invert=True)
        oxygen_scaled_cols.append(sc)

thermo_scaled_cols = []
for c in ["calcofi__t_degc", "calcofi__salnty", "calcofi__stheta"]:
    if c in aq.columns:
        sc = f"{c}__devscaled"
        aq[sc] = deviation_scale(aq[c])
        thermo_scaled_cols.append(sc)

mean_available(aq, bio_scaled_cols, "aq_component_nutrient_pressure")
mean_available(aq, oxygen_scaled_cols, "aq_component_oxygen_stress")
mean_available(aq, thermo_scaled_cols, "aq_component_thermo_shift")

weighted_mean_available(
    aq,
    [
        "aq_component_nutrient_pressure",
        "aq_component_oxygen_stress",
        "aq_component_thermo_shift",
    ],
    [0.45, 0.35, 0.20],
    "aq_component_biogeochemistry",
)

# ---------- component 2: nearshore contamination proxy ----------
contam_scaled_cols = []
for c in ["ca_beach__enterococcus", "ca_beach__total_coliform", "ca_beach__fecal_coliform"]:
    if c in aq.columns:
        sc = f"{c}__scaled"
        aq[sc] = quantile_scale(aq[c])
        contam_scaled_cols.append(sc)

mean_available(aq, contam_scaled_cols, "aq_component_contamination_proxy")

# ---------- component 3: hydrodynamics / coastal transport proxy ----------
hydro_scaled_cols = []

if "tides__water_level" in aq.columns:
    aq["tides__water_level__devscaled"] = deviation_scale(aq["tides__water_level"])
    hydro_scaled_cols.append("tides__water_level__devscaled")

if "tides__sigma" in aq.columns:
    aq["tides__sigma__scaled"] = quantile_scale(aq["tides__sigma"])
    hydro_scaled_cols.append("tides__sigma__scaled")

mean_available(aq, hydro_scaled_cols, "aq_component_hydrodynamics")

# ---------- freshness / confidence ----------
freshness_cols = []
for c in ["ca_beach__freshness_days", "tides__freshness_days"]:
    if c in aq.columns:
        sc = f"{c}__freshness_score"
        # fewer days old -> better freshness
        aq[sc] = quantile_scale(aq[c], invert=True)
        freshness_cols.append(sc)

mean_available(aq, freshness_cols, "aq_freshness_score")

if "n_sources_available" in aq.columns:
    max_sources = max(1, float(aq["n_sources_available"].max()))
    aq["aq_source_coverage_score"] = aq["n_sources_available"] / max_sources
else:
    aq["aq_source_coverage_score"] = np.nan

obs_support_cols = []
for c in ["ca_beach__obs_count", "tides__obs_count", "calcofi__obs_count"]:
    if c in aq.columns:
        sc = f"{c}__scaled"
        aq[sc] = quantile_scale(aq[c])
        obs_support_cols.append(sc)

mean_available(aq, obs_support_cols, "aq_observation_support_score")

weighted_mean_available(
    aq,
    ["aq_source_coverage_score", "aq_freshness_score", "aq_observation_support_score"],
    [0.4, 0.35, 0.25],
    "aq_confidence",
)

# ---------- final risk ----------
weighted_mean_available(
    aq,
    [
        "aq_component_biogeochemistry",
        "aq_component_contamination_proxy",
        "aq_component_hydrodynamics",
    ],
    [0.45, 0.35, 0.20],
    "aq_risk_score_raw",
)

aq = aq.sort_values(["site_id", "calendar_date"]).copy()
aq["aq_risk_score_smooth_3d"] = (
    aq.groupby("site_id")["aq_risk_score_raw"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

aq["aq_risk_score"] = np.where(
    aq["aq_risk_score_raw"].notna() | aq["aq_risk_score_smooth_3d"].notna(),
    0.6 * aq["aq_risk_score_raw"].fillna(aq["aq_risk_score_smooth_3d"]) +
    0.4 * aq["aq_risk_score_smooth_3d"].fillna(aq["aq_risk_score_raw"]),
    np.nan,
)

aq["aq_risk_level"] = risk_level_from_score(aq["aq_risk_score"])

aq["aq_top_drivers_json"] = aq.apply(
    lambda row: top_driver_json(
        row,
        {
            "biogeochemistry": row.get("aq_component_biogeochemistry"),
            "contamination_proxy": row.get("aq_component_contamination_proxy"),
            "hydrodynamics": row.get("aq_component_hydrodynamics"),
        },
    ),
    axis=1,
)

aq["aq_model_type"] = "heuristic_multisource_v1"

display_cols = [
    "site_id", "site_name", "region_name", "calendar_date",
    "aq_risk_score", "aq_risk_level", "aq_confidence",
    "aq_component_biogeochemistry",
    "aq_component_contamination_proxy",
    "aq_component_hydrodynamics",
    "aq_top_drivers_json",
]
display(aq[display_cols].head(20))

site_id,site_name,region_name,calendar_date,aq_risk_score,aq_risk_level,aq_confidence,aq_component_biogeochemistry,aq_component_contamination_proxy,aq_component_hydrodynamics,aq_top_drivers_json
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-02,0.34949209306040685,low,0.7914114469395955,0.5039828767509112,0.0,0.6134989926124841,"[{""driver"": ""hydrodynamics"", ""score"": 0.6135}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-03,0.35420653171716165,moderate,0.9914114469395955,0.5039828767509112,0.009161947519684369,0.6269308260577536,"[{""driver"": ""hydrodynamics"", ""score"": 0.6269}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0092}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-04,0.40707595109841027,moderate,0.8664114469395955,0.5039828767509112,4.336513443191674E-4,1.0,"[{""driver"": ""hydrodynamics"", ""score"": 1.0}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-05,0.3768044188901946,moderate,0.8164114469395954,0.5039828767509112,4.336513443191674E-4,0.7231922990821522,"[{""driver"": ""hydrodynamics"", ""score"": 0.7232}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-06,0.3957090900444262,moderate,0.7664114469395955,0.5039828767509112,4.336513443191674E-4,0.8373628833669113,"[{""driver"": ""hydrodynamics"", ""score"": 0.8374}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-07,0.41238545747969224,moderate,0.7164114469395955,0.5039828767509112,4.336513443191674E-4,0.9806357734497415,"[{""driver"": ""hydrodynamics"", ""score"": 0.9806}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-08,0.2805296258253389,low,0.6664114469395955,0.5039828767509112,4.336513443191674E-4,0.03481083501231623,"[{""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""hydrodynamics"", ""score"": 0.0348}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-09,0.3395699959469709,low,0.6164114469395955,0.5039828767509112,4.336513443191674E-4,0.5832773673606423,"[{""driver"": ""hydrodynamics"", ""score"": 0.5833}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-10,0.3082887482788871,low,0.5664114469395956,0.5039828767509112,4.336513443191674E-4,0.44224311618536183,"[{""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""hydrodynamics"", ""score"": 0.4422}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"
aq_agua_hedionda,Agua Hedionda Lagoon Watch,North San Diego County,2026-03-11,0.40095795206964857,moderate,0.5164114469395955,0.5039828767509112,4.336513443191674E-4,1.0,"[{""driver"": ""hydrodynamics"", ""score"": 1.0}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]"


In [0]:
with mlflow.start_run(run_name="aquaculture_risk_engine_v1") as run:
    mlflow.log_param("model_type", "heuristic_multisource_v1")
    mlflow.log_param("n_rows", len(aq))
    mlflow.log_param("n_sites", int(aq["site_id"].nunique()))
    mlflow.log_param("available_prefixes", json.dumps([k for k, v in prefix_summary.items() if len(v) > 0]))

    mlflow.log_metric("aq_rows_scored", float(aq["aq_risk_score"].notna().sum()))
    mlflow.log_metric("aq_mean_risk", float(pd.to_numeric(aq["aq_risk_score"], errors="coerce").mean()))
    mlflow.log_metric("aq_mean_confidence", float(pd.to_numeric(aq["aq_confidence"], errors="coerce").mean()))

    run_id = run.info.run_id

aq["mlflow_run_id"] = run_id

aq_out_cols = [
    "site_id", "site_name", "site_type", "region_name",
    "calendar_date", "lat_dec", "lon_dec", "tile_id",
    "aq_risk_score", "aq_risk_level", "aq_confidence",
    "aq_component_biogeochemistry",
    "aq_component_contamination_proxy",
    "aq_component_hydrodynamics",
    "aq_freshness_score", "aq_source_coverage_score", "aq_observation_support_score",
    "aq_top_drivers_json", "aq_model_type", "mlflow_run_id",
]

aq_out_pdf = aq[aq_out_cols].copy()
aq_out_sdf = to_spark_and_save(aq_out_pdf, f"{GOLD}.aquaculture_risk_daily")

display(aq_out_sdf)

site_id,site_name,site_type,region_name,calendar_date,lat_dec,lon_dec,tile_id,aq_risk_score,aq_risk_level,aq_confidence,aq_component_biogeochemistry,aq_component_contamination_proxy,aq_component_hydrodynamics,aq_freshness_score,aq_source_coverage_score,aq_observation_support_score,aq_top_drivers_json,aq_model_type,mlflow_run_id
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-02,33.14,-117.32,7378_-21872,0.34949209306040685,low,0.7914114469395955,0.5039828767509112,0.0,0.6134989926124841,0.4285714285714286,1.0,0.9656457877583821,"[{""driver"": ""hydrodynamics"", ""score"": 0.6135}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-03,33.14,-117.32,7378_-21872,0.35420653171716165,moderate,0.9914114469395955,0.5039828767509112,0.009161947519684369,0.6269308260577536,1.0,1.0,0.9656457877583821,"[{""driver"": ""hydrodynamics"", ""score"": 0.6269}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0092}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-04,33.14,-117.32,7378_-21872,0.40707595109841027,moderate,0.8664114469395955,0.5039828767509112,4.336513443191674E-4,1.0,1.0,1.0,0.46564578775838206,"[{""driver"": ""hydrodynamics"", ""score"": 1.0}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-05,33.14,-117.32,7378_-21872,0.3768044188901946,moderate,0.8164114469395954,0.5039828767509112,4.336513443191674E-4,0.7231922990821522,0.8571428571428572,1.0,0.46564578775838206,"[{""driver"": ""hydrodynamics"", ""score"": 0.7232}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-06,33.14,-117.32,7378_-21872,0.3957090900444262,moderate,0.7664114469395955,0.5039828767509112,4.336513443191674E-4,0.8373628833669113,0.7142857142857143,1.0,0.46564578775838206,"[{""driver"": ""hydrodynamics"", ""score"": 0.8374}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-07,33.14,-117.32,7378_-21872,0.41238545747969224,moderate,0.7164114469395955,0.5039828767509112,4.336513443191674E-4,0.9806357734497415,0.5714285714285714,1.0,0.46564578775838206,"[{""driver"": ""hydrodynamics"", ""score"": 0.9806}, {""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-08,33.14,-117.32,7378_-21872,0.2805296258253389,low,0.6664114469395955,0.5039828767509112,4.336513443191674E-4,0.03481083501231623,0.4285714285714286,1.0,0.46564578775838206,"[{""driver"": ""biogeochemistry"", ""score"": 0.504}, {""driver"": ""hydrodynamics"", ""score"": 0.0348}, {""driver"": ""contamination_proxy"", ""score"": 0.0004}]",heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,2026-03-09,33.14,-117.32,7378_-21872,0.3395699959469709,low,0.6164114469395955,0.5039828767509112,4.336513443191674E-4,0.5832773673606423,0.2857142857142857,1.0,0.46564578775838206,"[{""driver"": ""hydrodynamics"", ""score"": 0.5833}, {""drive

In [0]:
beach = beach_pdf.copy()

# Proxy event score from current bacteria measurements
bact_scaled_cols = []
for c in ["ca_beach__enterococcus", "ca_beach__total_coliform", "ca_beach__fecal_coliform"]:
    if c in beach.columns:
        sc = f"{c}__scaled"
        beach[sc] = quantile_scale(beach[c])
        bact_scaled_cols.append(sc)

if bact_scaled_cols:
    beach["beach_proxy_event_score"] = beach[bact_scaled_cols].max(axis=1, skipna=True)
    threshold = beach["beach_proxy_event_score"].quantile(0.75)
    beach["beach_proxy_label"] = (beach["beach_proxy_event_score"] >= threshold).astype(int)
else:
    beach["beach_proxy_event_score"] = np.nan
    beach["beach_proxy_label"] = 0

print("Beach proxy positives =", int(beach["beach_proxy_label"].sum()))
print("Beach proxy negatives =", int((1 - beach["beach_proxy_label"]).sum()))

Beach proxy positives = 7
Beach proxy negatives = 83


In [0]:
beach_model_run_id = None
beach_model_metrics = {}

# Predictors: non-bacteria environmental context + site metadata
candidate_feature_cols = []

for c in [
    "calcofi__depthm", "calcofi__t_degc", "calcofi__salnty", "calcofi__o2ml_l",
    "calcofi__stheta", "calcofi__o2sat", "calcofi__chlora", "calcofi__po4um",
    "calcofi__sio3um", "calcofi__no2um", "calcofi__no3um",
    "tides__water_level", "tides__sigma", "tides__freshness_days",
    "calcofi__avg_distance_km", "tides__min_distance_km",
    "n_sources_available", "site_priority",
]:
    if c in beach.columns:
        candidate_feature_cols.append(c)

beach_model_pdf = beach[
    ["site_id", "region_name", "calendar_date", "beach_proxy_label"] + candidate_feature_cols
].copy()

# Add simple categoricals
beach_X = pd.get_dummies(
    beach_model_pdf.drop(columns=["beach_proxy_label", "calendar_date"]),
    columns=["site_id", "region_name"],
    drop_first=False,
)

beach_y = beach_model_pdf["beach_proxy_label"].astype(int)

can_train = (
    len(beach_model_pdf) >= 20 and
    beach_y.nunique() == 2 and
    beach_y.sum() >= 5 and
    (len(beach_y) - beach_y.sum()) >= 5
)

if can_train:
    X_train, X_test, y_train, y_test = train_test_split(
        beach_X, beach_y,
        test_size=0.30,
        random_state=42,
        stratify=beach_y,
    )

    clf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])

    clf.fit(X_train, y_train)

    test_proba = clf.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= 0.5).astype(int)

    beach_model_metrics = {
        "roc_auc": float(roc_auc_score(y_test, test_proba)),
        "avg_precision": float(average_precision_score(y_test, test_proba)),
        "accuracy": float(accuracy_score(y_test, test_pred)),
        "precision": float(precision_score(y_test, test_pred, zero_division=0)),
        "recall": float(recall_score(y_test, test_pred, zero_division=0)),
        "f1": float(f1_score(y_test, test_pred, zero_division=0)),
    }

    with mlflow.start_run(run_name="beach_proxy_baseline_v1") as run:
        mlflow.log_param("model_type", "logistic_regression_proxy_baseline")
        mlflow.log_param("n_rows", len(beach_model_pdf))
        mlflow.log_param("n_features", beach_X.shape[1])
        mlflow.log_param("feature_cols", json.dumps(list(beach_X.columns)))

        for k, v in beach_model_metrics.items():
            mlflow.log_metric(k, v)

        try:
            mlflow.sklearn.log_model(clf, artifact_path="beach_proxy_model")
        except Exception as e:
            print("Could not log sklearn model artifact:", e)

        beach_model_run_id = run.info.run_id

    beach["beach_risk_score"] = clf.predict_proba(beach_X)[:, 1]
    beach["beach_model_type"] = "logistic_regression_proxy_baseline"

else:
    print("Skipping beach supervised baseline; insufficient class support.")
    beach["beach_risk_score"] = beach["beach_proxy_event_score"]
    beach["beach_model_type"] = "heuristic_proxy_fallback"

2026/04/19 17:53:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-e924a92a-29d9.cloud.databricks.com/ml/experiments/4339572085984185/models/m-b53c431d0df046f28036541d92b884ce?o=7474649761674786
2026/04/19 17:53:16 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


In [0]:
# confidence for beach
freshness_cols = []
for c in ["ca_beach__freshness_days", "tides__freshness_days"]:
    if c in beach.columns:
        sc = f"{c}__freshness_score"
        beach[sc] = quantile_scale(beach[c], invert=True)
        freshness_cols.append(sc)

if freshness_cols:
    beach["beach_freshness_score"] = beach[freshness_cols].mean(axis=1, skipna=True)
else:
    beach["beach_freshness_score"] = np.nan

if "n_sources_available" in beach.columns:
    max_sources = max(1, float(beach["n_sources_available"].max()))
    beach["beach_source_coverage_score"] = beach["n_sources_available"] / max_sources
else:
    beach["beach_source_coverage_score"] = np.nan

weighted_mean_available(
    beach,
    ["beach_freshness_score", "beach_source_coverage_score"],
    [0.45, 0.55],
    "beach_confidence",
)

beach["beach_risk_level"] = risk_level_from_score(beach["beach_risk_score"])

beach["beach_top_drivers_json"] = beach.apply(
    lambda row: top_driver_json(
        row,
        {
            "enterococcus_proxy": row.get("ca_beach__enterococcus__scaled"),
            "fecal_coliform_proxy": row.get("ca_beach__fecal_coliform__scaled"),
            "total_coliform_proxy": row.get("ca_beach__total_coliform__scaled"),
            "tide_state": quantile_scale(pd.Series([row.get("tides__water_level")])).iloc[0]
                if "tides__water_level" in beach.columns else np.nan,
        },
    ),
    axis=1,
)

beach["mlflow_run_id"] = beach_model_run_id

beach_out_cols = [
    "site_id", "site_name", "site_type", "region_name",
    "calendar_date", "lat_dec", "lon_dec", "tile_id",
    "beach_proxy_label", "beach_proxy_event_score",
    "beach_risk_score", "beach_risk_level", "beach_confidence",
    "beach_top_drivers_json", "beach_model_type", "mlflow_run_id",
]

beach_out_pdf = beach[beach_out_cols].copy()
beach_out_sdf = to_spark_and_save(beach_out_pdf, f"{GOLD}.beach_risk_daily")

display(beach_out_sdf)

site_id,site_name,site_type,region_name,calendar_date,lat_dec,lon_dec,tile_id,beach_proxy_label,beach_proxy_event_score,beach_risk_score,beach_risk_level,beach_confidence,beach_top_drivers_json,beach_model_type,mlflow_run_id
beach_mission_beach,Mission Beach,beach,San Diego,2026-03-03,32.77,-117.25,7295_-21951,1,1.0,0.9544879073776584,severe,0.767241379310345,"[{""driver"": ""enterococcus_proxy"", ""score"": 1.0}]",logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_mission_beach,Mission Beach,beach,San Diego,2026-03-21,32.77,-117.25,7295_-21951,0,null,0.06098742830589355,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_coronado,Coronado Central Beach,beach,San Diego,2026-03-06,32.68,-117.18,7275_-21960,0,0.8554181676565086,0.09517109796411166,low,0.767241379310345,"[{""driver"": ""enterococcus_proxy"", ""score"": 0.8554}]",logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_coronado,Coronado Central Beach,beach,San Diego,2026-03-13,32.68,-117.18,7275_-21960,0,null,4.3580915209107747E-4,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_coronado,Coronado Central Beach,beach,San Diego,2026-03-18,32.68,-117.18,7275_-21960,0,null,3.035469149037031E-4,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_coronado,Coronado Central Beach,beach,San Diego,2026-03-24,32.68,-117.18,7275_-21960,0,null,2.9026825394025136E-4,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_coronado,Coronado Central Beach,beach,San Diego,2026-03-29,32.68,-117.18,7275_-21960,0,null,3.0589188319401E-4,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_coronado,Coronado Central Beach,beach,San Diego,2026-03-30,32.68,-117.18,7275_-21960,0,null,3.6459275052789445E-4,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_mission_beach,Mission Beach,beach,San Diego,2026-03-06,32.77,-117.25,7295_-21951,1,1.0,0.9582595125124586,severe,0.55,"[{""driver"": ""enterococcus_proxy"", ""score"": 1.0}]",logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280
beach_mission_beach,Mission Beach,beach,San Diego,2026-03-09,32.77,-117.25,7295_-21951,0,null,0.057372179768535064,low,0.5,[],logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280


In [0]:
summary_rows = [
    {
        "model_name": "aquaculture_risk_engine_v1",
        "model_type": "heuristic_multisource_v1",
        "mlflow_run_id": aq["mlflow_run_id"].iloc[0] if len(aq) > 0 else None,
        "n_rows": int(len(aq)),
        "primary_metric_name": "mean_risk",
        "primary_metric_value": float(pd.to_numeric(aq["aq_risk_score"], errors="coerce").mean()),
    },
    {
        "model_name": "beach_proxy_baseline_v1" if beach_model_run_id else "beach_proxy_fallback_v1",
        "model_type": beach["beach_model_type"].iloc[0] if len(beach) > 0 else None,
        "mlflow_run_id": beach_model_run_id,
        "n_rows": int(len(beach)),
        "primary_metric_name": "roc_auc" if "roc_auc" in beach_model_metrics else "mean_score",
        "primary_metric_value": float(
            beach_model_metrics["roc_auc"] if "roc_auc" in beach_model_metrics
            else pd.to_numeric(beach["beach_risk_score"], errors="coerce").mean()
        ),
    },
]

model_run_summary_pdf = pd.DataFrame(summary_rows)
model_run_summary_sdf = to_spark_and_save(model_run_summary_pdf, f"{GOLD}.model_run_summary")
display(model_run_summary_sdf)

model_name,model_type,mlflow_run_id,n_rows,primary_metric_name,primary_metric_value
aquaculture_risk_engine_v1,heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b,180,mean_risk,0.4754124263789888
beach_proxy_baseline_v1,logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280,90,roc_auc,0.94


In [0]:
aq_sdf = spark.read.table(f"{GOLD}.aquaculture_risk_daily")
beach_sdf = spark.read.table(f"{GOLD}.beach_risk_daily")
summary_sdf = spark.read.table(f"{GOLD}.model_run_summary")

print("aquaculture_risk_daily rows =", aq_sdf.count())
print("beach_risk_daily rows =", beach_sdf.count())
print("model_run_summary rows =", summary_sdf.count())

display(
    aq_sdf.groupBy("site_name", "aq_risk_level")
    .count()
    .orderBy("site_name", "aq_risk_level")
)

display(
    beach_sdf.groupBy("site_name", "beach_risk_level")
    .count()
    .orderBy("site_name", "beach_risk_level")
)

display(summary_sdf)

aquaculture_risk_daily rows = 180
beach_risk_daily rows = 90
model_run_summary rows = 2


site_name,aq_risk_level,count
Agua Hedionda Lagoon Watch,high,1
Agua Hedionda Lagoon Watch,low,4
Agua Hedionda Lagoon Watch,moderate,25
Humboldt Bay Shellfish Watch,severe,30
Mission Bay Shellfish Watch,high,9
Mission Bay Shellfish Watch,moderate,21
Morro Bay Shellfish Watch,low,30
Newport Bay Shellfish Watch,low,30
San Diego Bay Shellfish Watch,high,7
San Diego Bay Shellfish Watch,moderate,19


site_name,beach_risk_level,count
Coronado Central Beach,low,30
La Jolla Shores,high,7
La Jolla Shores,low,20
La Jolla Shores,severe,3
Mission Beach,low,25
Mission Beach,severe,5


model_name,model_type,mlflow_run_id,n_rows,primary_metric_name,primary_metric_value
aquaculture_risk_engine_v1,heuristic_multisource_v1,d04a2d77c01d4e26b13e6d0f3d9ea15b,180,mean_risk,0.4754124263789888
beach_proxy_baseline_v1,logistic_regression_proxy_baseline,59d8145a10a64210bfcbeeb3ff550280,90,roc_auc,0.94
